# GPT from scratch — Part 3: Multi-head attention → the transformer block

Continues from `single-head-attention.ipynb`. A single causal head, trained honestly with
`estimate_loss`, plateaued at **~2.44** — barely past the bigram's 2.50. The diagnosis: attention
*gathers* information between positions, but nothing *computes* on what's gathered, and one head
can only track one kind of relationship at a time.

**Roadmap this notebook (one concept per step):**
1. **Multi-head attention** — several heads in parallel, each learning a different relationship, concatenated.
2. **Feed-forward** — per-token computation on the gathered context (the missing "thinking" step).
3. **Residual connections + LayerNorm** — what makes a deep stack actually trainable.
4. **The transformer block** — attention + feed-forward wrapped into one repeatable unit.
5. **Scale up** — bigger `n_embd`, more blocks, longer context → real Shakespeare.

Setup + the `Head` module + `estimate_loss` are carried over below — run top-to-bottom, then we build.

## Setup (carried over from Parts 1–2)

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)

torch 2.12.1 | device: mps


In [3]:
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
print('length:', len(text), '| vocab_size:', vocab_size)

length: 1115394 | vocab_size: 65


In [4]:
# Tokenizer (from Part 1)
stoi = {c: n for n, c in enumerate(chars)}
itos = {n: c for n, c in enumerate(chars)}

def encode(s):
    return list(map(lambda c: stoi[c], s))

def decode(l):
    return ''.join(map(lambda i: itos[i], l))

assert decode(encode('hii there')) == 'hii there'

In [5]:
# Corpus -> tensor, 90/10 train/val split (from Part 1)
data = torch.tensor(encode(text), dtype=torch.int64)
n = int(0.9 * data.shape[0])
train_data = data[:n]
val_data = data[n:]
print('train:', len(train_data), '| val:', len(val_data))

train: 1003854 | val: 111540


In [6]:
# Hyperparameters + batching (from Parts 1–2)
block_size = 8    # T: max context length
batch_size = 4    # B: independent sequences per batch
n_embd = 32       # embedding dimension ("C")

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, d.shape[0] - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('xb', xb.shape, '| yb', yb.shape)

xb torch.Size([4, 8]) | yb torch.Size([4, 8])


In [7]:
# The single causal head from Part 2 (your working version) -- the building block we now parallelize.
class Head(nn.Module):
    """ One head of causal self-attention. """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(1, 2) * (self.head_size ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, 2)
        out = wei @ v
        return out

In [8]:
# estimate_loss from Part 2 (your version) -- stable train/val loss, averaged over many batches.
eval_iters = 200

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ('train', 'val'):
        losses = torch.zeros((eval_iters,))
        for i in range(eval_iters):
            _, losses[i] = model(*get_batch(split))
        out[split] = losses.mean()
    model.train()
    return out

## Step 1 — Multi-head attention

One head learns **one** kind of relationship — maybe "attend to the previous vowel," maybe
"attend to the last space." Language needs many of these at once. **Multi-head** attention is the
embarrassingly simple fix: run `num_heads` independent heads *in parallel*, each with its own
q/k/v projections, then **concatenate** their outputs so the next layer sees all of them.

**The sizing trick:** we set `head_size = n_embd // num_heads`. Four heads of size 8 concatenate
back to `4 * 8 = 32 = n_embd`. That keeps the tensor width constant across the layer — which
matters a lot in Step 3, when we start *adding* this output back onto its input (residuals need
matching shapes).

**One PyTorch wrinkle (given in the code, but understand it):** the heads live in an
`nn.ModuleList`, not a plain Python list. `nn.ModuleList` *registers* each head as a submodule, so
its parameters show up in `model.parameters()`, get gradients, get trained, and move with
`.to(device)`. A plain `[Head(...), Head(...)]` would be **invisible** to all of that — no error,
the model just silently never trains those weights. It's the same family of trap as passing the
*class* instead of the *instance* to the optimizer back in Part 1: PyTorch only tracks what you
register through its machinery.

**Your turn** is the `forward` — the parallel-then-concatenate move. Think about *which axis* you
concatenate along, and why concatenating `(B,T,head_size)` tensors along it reassembles the full
`(B,T,n_embd)` width.

In [10]:
# Step 1 — multi-head attention as a reusable layer.
class MultiHeadAttention(nn.Module):
    """ num_heads causal self-attention heads, run in parallel and concatenated. """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # GIVEN (the wrinkle): ModuleList registers each Head so its params are tracked,
        # trained, and moved to device. A plain list would silently hide them.
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        # TODO(human): run x through EVERY head, then concatenate the outputs along the feature axis.
        #   - each head(x) returns (B, T, head_size)
        #   - concatenating num_heads of them along the last dim gives (B, T, num_heads*head_size)
        #   - since head_size = n_embd // num_heads, that lands back at (B, T, n_embd)
        #   tools: torch.cat(list_of_tensors, dim=?)  and a list comprehension over self.heads.
        #   which dim is the feature axis of a (B, T, hs) tensor?

        return torch.cat([head(x) for head in self.heads], dim=-1)

## insight
# we are using the single attention head from the other notebook. Where instead of AttentionLM feeding
# positions and embeddings directly into the single head, we are putting it into this MultiHeadAttention module
# that stores a list of the attention heads, and handles them in parallel where AttentionLM interacts with MultiHead
# as if it was a single head.

# --- Smoke test: 4 heads of size 8 -> concat back to n_embd = 32 ---
torch.manual_seed(1337)
x = torch.randn(4, block_size, n_embd)          # (B=4, T=8, n_embd=32)
mha = MultiHeadAttention(num_heads=4, head_size=n_embd // 4)
out = mha(x)
print('out', out.shape)                         # expect (4, 8, 32): four (4,8,8) heads concatenated
print('finite?', torch.isfinite(out).all().item())

out torch.Size([4, 8, 32])
finite? True


In [16]:
# Step 1 (cont.) — wire multi-head into the LM and train. Do 4 heads beat the 2.44 single-head plateau?
# GIVEN: this is Part 2's AttentionLM with a ONE-LINE swap (Head -> MultiHeadAttention). The concept
# work was the concat you just wrote; seeing the loss move is the reward.
class AttentionLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_embd // 4)  # <- was Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)
        pos = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.sa_heads(x)                      # four heads in parallel, concatenated back to n_embd
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


torch.manual_seed(1337)
model = AttentionLM().to(device)
print('params:', sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5000):
    if step % 500 == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\\n--- sample ---')
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))


params: 7553
step    0 | train 4.2254 | val 4.2241
step  500 | train 2.9213 | val 2.9500
step 1000 | train 2.7325 | val 2.7376
step 1500 | train 2.6140 | val 2.6342
step 2000 | train 2.5727 | val 2.5971
step 2500 | train 2.5306 | val 2.5463
step 3000 | train 2.4669 | val 2.4847
step 3500 | train 2.4447 | val 2.4554
step 4000 | train 2.4289 | val 2.4403
step 4500 | train 2.4051 | val 2.3950
\n--- sample ---

Tho. I bhe herat so toundes oKou w sound ghid.a
DUGAPLANSTIU:
IN tom, se heaf dorid:
W: fort tour keapin dis dut whe by ant poon mid:
Nur baneinge oss Vaby crove por fave
Thetes thexm-ald thissern, ant sE
MCENTINT mave

for de faur; ro shat it shit arlit I ma the ot,
Aniecet wo mt se ay ain
To: hat and now de sGb
Somy lom I de ma byoui, der bund ofald dy baive wisor thencu gher thid an tarthet youiug thessth crif paune yy wous iththere of me ly be of of tour srid de'ldeme.

D th magn;
Toucicros niy souse thhe Wom rencsts nwo fuf hout chatind de a thy natenge, ay he, ft se bic?' ass 

In [17]:
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))


Th.ielind pandus kok fors dy do mearlese uf.
LRGCAOIF
AWno ralt so ha shenas dedenlt, tha ardy may I deUThy. 'sth lor youli thwer tho ot de pcor:
De bts,
Spt wanvingt che aticet deburotfnsie he, ware the man Imtu fert my be
INb:
Yi, wenod sot to whe qur why dish taste, why lif and dus the wen thithe nod sad doth 'ls ragnPir in.

Po whrirl, wou, us te dinstheas maudl a, moKo neaurg doure:
Athey, paobaot beaty semy dag
Kthe neg thot.

Bep;
A-nomo, Kins!
Ay, yreds I forto
Ted twind.

Mor den cho and did;
Lhe
She ner why fretour' fcoocwuted obet kinte nomve shi, tortt e the anescth

Tone toucroungor meme by I Io mas co tharl mazast how fait not bin'ld I I houd; tho as e tho.

Pthrand re

Lfereend.

A:au wibouh thy thereld the dik cpateshe sho's of an at!
aor onciethous,
IETEY:
Se thawpes the deairt sine thed:

I dow Oyoud.

ASCK
CVHT:
AUOLome this en parct tho dof belt loy se thers wos cidred sos isthe
astt dot vand.
Cng wot? 'f fho is for he fre pou woer.

BI was opngie, the?
Cot si thar

## Insights

Having 4 heads generates text with more structure to it. It is not generating intelligible words, but the spacing and structure is starting to look a lot more like the sample text (english) compared to a single attention head

```txt
Th.ielind pandus kok fors dy do mearlese uf.
LRGCAOIF
AWno ralt so ha shenas dedenlt, tha ardy may I deUThy. 'sth lor youli thwer tho ot de pcor:
De bts,
Spt wanvingt che aticet deburotfnsie he, ware the man Imtu fert my be
INb:
Yi, wenod sot to whe qur why dish taste, why lif and dus the wen thithe nod sad doth 'ls ragnPir in.

Po whrirl, wou, us te dinstheas maudl a, moKo neaurg doure:
Athey, paobaot beaty semy dag
Kthe neg thot.

Bep;
A-nomo, Kins!
Ay, yreds I forto
Ted twind.

Mor den cho and did;
Lhe
She ner why fretour' fcoocwuted obet kinte nomve shi, tortt e the anescth

Tone toucroungor meme by I Io mas co tharl mazast how fait not bin'ld I I houd; tho as e tho.

Pthrand re
...
Cng wot? 'f fho is for he fre pou woer.
```

Having 8 heads doesn't make much of a discernable difference

```txt
and now dousGb
Somy whind be ma boou ardy hound olal tharepis wo so no, peu garthe that a fith thy sit, tho sth crif pat hoyy, thou tathere ora they be of of tou thris axindeme.

Do,
SSe not macicr:
And whoise thalevy are, sts nwo fuf hant chatine dacd at that-nge, ay fu, fer do, '?' sas nonst cord caflid
Thene
Frd
Te!
IfUSt ineordlded hof wo? I corchepgengt, at wes coin;
Lout for youhisst to sbuns to that yedimr as?
ATe:
CL
AThatiten
Vont torand-bm thabikc:' tof o douth er tou nouce:
Hige you omot.
ARon
LOIY mor's doucho lamy ale t bout;
Adinde fou ILCe shame themit at. Wher Coul fi, stte, and mear b
Techres pust viske; mo thy sem fry ny aigs een I him, han of of so hous I nge bour, s thu, nipieth bepandus kok fous th the darlese fand my
And do
Ao ralt se ha shenas dooenld, tin arfy may! al UF.
I I INNCo:
The pit wof tho of da pime:
...
Ting
Houd dou my be
INb
Ati, wenor sot to whe qur'd'g dis utat'nd, figou uadl dusmt; feen thing:
```

In [ ]:
# --- 4 heads vs 8 heads: compare the LOSS CURVES, not the vibes. ---
# Eyeballing generated text is noisy (multinomial sampling). The val-loss curve is the
# apples-to-apples measurement. head_size = n_embd // num_heads, so total width is FIXED at
# n_embd either way: 4 heads of size 8, or 8 heads of size 4. Same capacity, partitioned differently.
import matplotlib.pyplot as plt

class ConfigurableAttentionLM(nn.Module):
    """ Same as AttentionLM, but num_heads is a knob so we can sweep it. """
    def __init__(self, num_heads):
        super().__init__()
        assert n_embd % num_heads == 0, "num_heads must divide n_embd evenly (concat must land back at n_embd)"
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads, n_embd // num_heads)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.sa_heads(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

@torch.no_grad()
def val_loss(m):                        # like estimate_loss, but takes the model explicitly (val only)
    m.eval()
    losses = torch.zeros(eval_iters)
    for i in range(eval_iters):
        _, losses[i] = m(*get_batch('val'))
    m.train()
    return losses.mean().item()

def train_and_track(num_heads, steps=3000, eval_every=250):
    torch.manual_seed(1337)             # same weight-init origin for both runs
    m = ConfigurableAttentionLM(num_heads).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    torch.manual_seed(2024)             # RESEED here so both runs see the SAME batch stream ->
                                        # any curve difference is architecture, not luck-of-the-draw
    steps_x, loss_y = [], []
    for step in range(steps):
        if step % eval_every == 0:
            steps_x.append(step)
            loss_y.append(val_loss(m))
        xb, yb = get_batch('train')
        _, loss = m(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    return steps_x, loss_y

# Two runs (takes ~1-2 min on MPS: 2 models x 3000 steps + periodic eval).
x4, y4 = train_and_track(4)
x8, y8 = train_and_track(8)

plt.figure(figsize=(8, 5))
plt.plot(x4, y4, marker='o', ms=3, label='4 heads x size 8')
plt.plot(x8, y8, marker='s', ms=3, label='8 heads x size 4')
plt.xlabel('training step')
plt.ylabel('val loss (avg over 200 batches)')
plt.title(f'Multi-head sweep (n_embd={n_embd}, block_size={block_size})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f'final val  |  4 heads: {y4[-1]:.4f}   8 heads: {y8[-1]:.4f}')


## Step 2 — The feed-forward layer (finally, computation)

The plateau has been telling you the same thing since single-head: **attention only *gathers*.**
Every token now holds a context vector assembled from its past — but nothing has *processed* that
vector. The model is all communication, no computation.

The feed-forward layer *is* the computation: a tiny per-token MLP. After attention mixes
information **across positions**, the feed-forward mixes information **across features** — inside
each position, independently (position 3's MLP never looks at position 5).

**The two halves of a transformer, stated plainly:**
- **Attention = communication.** Tokens talk to each other (along the `T` axis).
- **Feed-forward = computation.** Each token thinks about what it heard (along the `C` axis,
  one position at a time).

**Shape journey:** `n_embd → (wider hidden) → n_embd`. In and out are both `(B,T,n_embd)`, so it
drops straight into the residual stream in Step 3. The expansion in the middle (the paper uses
**4×**) gives the MLP room to compute a richer intermediate before compressing back down.

**Why a nonlinearity is non-negotiable:** stack two `Linear`s with nothing between them and they
collapse — `W₂(W₁x) = (W₂W₁)x`, algebraically *one* linear layer. A **ReLU** between them breaks
the collapse; it's the entire reason two matrices can compute something one matrix can't. No
nonlinearity, no thinking.

**New idiom — `nn.Sequential`:** a container that chains modules and, on `forward`, just pipes the
input through them in order. It's `nn.ModuleList`'s tidier cousin for the plain "A then B then C"
case — no manual loop, params still registered and trained.

**Your turn** is the `__init__` — the shape of the MLP itself: expand, nonlinear, project back.

In [19]:
# Step 2 — the feed-forward layer: a per-token MLP.
class FeedForward(nn.Module):
    """ Attention gathered the context; this COMPUTES on it, each position independently. """

    def __init__(self, n_embd):
        super().__init__()
        # TODO(human): build a small MLP as an nn.Sequential, stored in self.net.
        #   1) EXPAND: a Linear from n_embd up to a wider hidden dim. The transformer paper uses 4*n_embd.
        #   2) NONLINEARITY: nn.ReLU()  -- without it, two stacked Linears collapse to one (W2 @ W1),
        #      so the whole layer would be linear and compute nothing new.
        #   3) PROJECT BACK: a Linear from the hidden dim back down to n_embd, so the output width
        #      matches the input (it has to slot into the residual stream in Step 3).
        #   nn.Sequential(a, b, c) chains modules and pipes x through them in order on forward().
        # self.net = ...
        self.net = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.ReLU(), nn.Linear(4*n_embd, n_embd))

    def forward(self, x):
        return self.net(x)      # (B, T, n_embd) -> (B, T, n_embd), applied per position


# --- Smoke test: same width in and out ---
torch.manual_seed(1337)
x = torch.randn(4, block_size, n_embd)      # (B=4, T=8, n_embd=32)
ff = FeedForward(n_embd)
out = ff(x)
print('out', out.shape)                     # expect (4, 8, 32) -- unchanged width
print('finite?', torch.isfinite(out).all().item())


out torch.Size([4, 8, 32])
finite? True


In [21]:
# Step 2 (cont.) — wire feed-forward into the LM and train. Does "compute" finally break 2.4?
# GIVEN: AttentionLM again, now with ONE new line -- self.ffwd -- applied right after attention.
# communication (sa_heads) THEN computation (ffwd), then decode (lm_head).
class AttentionLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_embd // 4)
        self.ffwd = FeedForward(n_embd)                    # <- NEW: the per-token computation
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.sa_heads(x)                                # communication: tokens gather from the past
        x = self.ffwd(x)                                    # computation: each token processes what it gathered
        logits = self.lm_head(x)                            # decode: -> vocab logits
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


torch.manual_seed(1337)
model = AttentionLM().to(device)
print('params:', sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5000):
    if step % 500 == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\\n--- sample ---')
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


params: 15905
step    0 | train 4.1538 | val 4.1495
step  500 | train 2.8612 | val 2.7821
step 1000 | train 2.6461 | val 2.6184
step 1500 | train 2.5597 | val 2.5373
step 2000 | train 2.5129 | val 2.5029
step 2500 | train 2.4415 | val 2.4244
step 3000 | train 2.4349 | val 2.4158
step 3500 | train 2.3928 | val 2.3865
step 4000 | train 2.3694 | val 2.3586
step 4500 | train 2.3373 | val 2.3499
\n--- sample ---

BRWl and I herat sudt youf,
DO Shi sours ghideariim you nit ton st mise wist woreid:
Wid,
Shave hakeap.

Nis.


Tous be min poon mid:
Nur banes ank.
MIVYREF to n pos fave
The as thexmadlive is is my bor EBall, sot mave?
Nor Sernd your malkinicull billl than a the of,
Wo therse cit or alvely
To shat and nush borG I omy lound be mavend. OJTOOOUSAR:
Walled keaive wisore yrouu gher thileall fith thy siug, is you crif palle.

NO-NEAN:
Sore borall lyour of of tour srid alll ben.

Dot Salgl;
Tour croat


## Step 3 — Residual connections + LayerNorm (making depth trainable)

Your `forward` is now `attention → feed-forward → decode` — one "communicate–compute" pair. The
obvious next move is to **stack** several so the model can compose (attention over the results of
attention). But naively stacking `x = sublayer(x)` several deep **won't train** — gradients vanish
or explode on the way back and the loss just sits there. Two mechanisms fix this; every deep net
uses both.

**1. Residual connections.** Instead of `x = sublayer(x)`, write `x = x + sublayer(x)`. The sublayer
no longer *replaces* the stream — it computes a **delta** added onto it. Two payoffs:
- **Gradient highway.** Backprop through `x + f(x)` sends `1 + f'(x)` backward — that `1` is a clean,
  unattenuated path to earlier layers, so even a deep stack keeps a strong gradient. This is *the*
  trick that let networks get deep (ResNet, 2015).
- **Easy identity.** A block can learn to do almost nothing (drive `sublayer(x) → 0`) and just pass
  `x` along. Starting near-identity and learning small refinements optimizes far more easily than
  learning a full transformation at every layer.

*(This is why we held every sublayer at width `n_embd` — `x` and `sublayer(x)` must match shape to add.)*

**2. LayerNorm.** As data flows through many layers, activations drift — means wander, variances blow
up or collapse. `nn.LayerNorm(n_embd)` renormalizes **each token's feature vector** to mean 0, std 1
(over the `C` axis, per position, per sample), then scales/shifts by two learned parameters. Every
layer gets a sanely-scaled input regardless of depth.
- **Vs BatchNorm:** LayerNorm normalizes across *features within one token*, independent of the other
  tokens in the batch — so it doesn't care about batch size and behaves *identically* in train and
  eval. (Graphics analogy: normalize each pixel across its own channels, not across the whole image.)

**Pre-norm placement.** Modern transformers apply the norm *before* the sublayer: `x = x + sublayer(ln(x))`.
(The original paper normalized *after*; pre-norm trains more stably, so that's what we build.)

Put together, one **block** is exactly two lines:
```
x = x + attention(   ln1(x) )     # communicate, residual + pre-norm
x = x + feedforward( ln2(x) )     # compute,     residual + pre-norm
```

That two-line `forward` is the beating heart of the transformer. **Your turn** to write it — the
`__init__` (two sublayers, two LayerNorms) is given.

In [22]:
# Step 3 — the transformer Block: attention + feed-forward, each wrapped in a residual + pre-norm.
class Block(nn.Module):
    """ Communicate (attention) then compute (feed-forward), each with a residual and pre-LayerNorm. """

    def __init__(self, n_embd, num_heads):
        super().__init__()
        self.sa   = MultiHeadAttention(num_heads, n_embd // num_heads)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)      # normalize each token's features BEFORE attention
        self.ln2  = nn.LayerNorm(n_embd)      # ... and BEFORE the feed-forward

    def forward(self, x):
        # TODO(human): two residual connections, \"pre-norm\" style.
        #   1) x = x + <self.sa   applied to self.ln1(x)>
        #   2) x = x + <self.ffwd applied to self.ln2(x)>
        #   The \"x = x + ...\" IS the residual: each sublayer computes a DELTA that gets ADDED onto the
        #   stream (it refines x, doesn't replace it). That '+' is a gradient highway -- gradient flows
        #   straight back through it, so a deep stack of these still trains.
        #   ln1/ln2 normalize each token's C-vector (mean 0, std 1) BEFORE its sublayer (pre-norm),
        #   keeping activations well-scaled no matter how deep the stack.
        # return x
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# --- Smoke test: a block preserves shape (so blocks can be stacked) ---
torch.manual_seed(1337)
x = torch.randn(4, block_size, n_embd)
block = Block(n_embd, num_heads=4)
out = block(x)
print('out', out.shape)           # expect (4, 8, 32) -- same width in and out
print('finite?', torch.isfinite(out).all().item())


out torch.Size([4, 8, 32])
finite? True


## Step 4 — Stack the blocks (depth, at last)

Your `Block` takes `(B,T,n_embd)` in and gives `(B,T,n_embd)` out — that shape-preservation is
*exactly* what makes it stackable. Chain several and the model can build relationships **on top of**
relationships: block 2's attention operates on what block 1 already refined, block 3 on that, and
so on. Depth is where a transformer gets its real power — and it only works because Step 3's
residuals keep the gradients alive all the way down.

**Two given pieces:**
- **`self.blocks(x)`** in `forward` — one call runs the input through the entire stack in order.
- **`self.ln_f = nn.LayerNorm(n_embd)`** — a final norm right before `lm_head`, so the decoder always
  sees clean, well-scaled features no matter how deep the stack. (Standard in GPT.)

**Your turn** is one line: build `self.blocks` as a stack of `n_layer` `Block`s.

**What to watch when it trains:** with ~3× the depth and the most parameters yet, expect the loss to
drop further — *and* expect the `train`/`val` gap to finally open up noticeably. That gap is
**overfitting** starting to bite: the model is now big enough to memorize training quirks that don't
generalize. That's not a failure — it's the signal that tells you it's time for **Step 5**, where
scaling up brings in **dropout** (and where your `model.eval()` habit finally does real work).

In [26]:
# Step 4 — stack the blocks into a deep model, then train.
class AttentionLM(nn.Module):
    def __init__(self, n_layer=3, num_heads=4):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # TODO(human): build self.blocks -- a stack of n_layer Block(n_embd, num_heads) modules
        #   run in sequence. Same container as FeedForward's self.net (nn.Sequential), but built
        #   from a LIST of n_layer blocks. How do you turn a Python list into the separate
        #   positional args nn.Sequential expects? (the * unpacking operator + a list comprehension)
        # self.blocks = ...
        self.blocks = nn.Sequential(*[Block(n_embd, num_heads) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)              # final norm before decoding (given)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.blocks(x)                               # run through the whole stack of blocks
        x = self.ln_f(x)                                 # final LayerNorm
        logits = self.lm_head(x)                         # decode -> vocab logits
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


torch.manual_seed(1337)
model = AttentionLM(n_layer=3, num_heads=4).to(device)
print('params:', sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5000):
    if step % 500 == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\\n--- sample ---')
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


params: 39201
step    0 | train 4.3315 | val 4.3347
step  500 | train 2.6752 | val 2.7131
step 1000 | train 2.4940 | val 2.5315
step 1500 | train 2.4508 | val 2.4412
step 2000 | train 2.4200 | val 2.3807
step 2500 | train 2.3351 | val 2.4035
step 3000 | train 2.3288 | val 2.3382
step 3500 | train 2.2771 | val 2.2988
step 4000 | train 2.2948 | val 2.3090
step 4500 | train 2.2683 | val 2.2889
\n--- sample ---

Tholling!
Therat so to VOforomy hi so$n rothe.

DUGAPLANSTIUS:
MINGSike wist woreid:
Wiand therurek-apy shas dunce,
Wint in a:
G mot:
Nur banes and prany
Cact bu ples.

SUSLALETS:
Cy thoust in ifn, a prows me, son mave
Slat Serny your magat in so binglm: I mange to as ofeert reast or a suly
To she gand now do swaxsomy lomer if mavend! and romencrolalbed in is worsore you ufeld wery saAn:
Thorery sour to so macreffpan hoyy,
-dond athery ora they bacond forour srigh.
ODINGFLOM:
Dp magn;
To cIYroat


## Step 5 — Scale up + dropout (the finale)

Two things happen here, and **only one is a new idea.**

### The dials (not new — just bigger numbers)
The architecture is *finished*. Everything from here is turning knobs: longer context, wider
embeddings, more blocks, more heads, bigger batches. The biggest single lever is **`block_size`
(context)** — at 8 the model can't see a whole word; the jump to real English lives mostly here.
This is the scaling hypothesis made concrete: GPT-3 is *this notebook* with the dials cranked.

### Dropout (the one new mechanism)
In Step 4 you watched the train/val gap open — the model began *memorizing*. Dropout is the direct
fix. During **training**, it randomly zeroes a fraction `p` of activations on every forward pass
(and scales the survivors by `1/(1-p)` to keep the expected magnitude). During **eval**, it does
nothing — all units active.

Why it regularizes: the network can't lean on any single unit or connection, because that unit
might vanish next step. It's forced to spread its bets and learn **redundant, robust** features —
effectively training a huge ensemble of subnetworks that share weights, then averaging them at eval.

**This is the payoff for your `model.eval()` discipline.** `eval()` is precisely what switches
dropout **off**. Every `estimate_loss` and `generate` call has been quietly calling it so you
measured and sampled the *full* network — not a randomly lobotomized one. Until now it was a no-op
habit; with dropout it is **load-bearing**. Forget it and your val loss goes noisy and your samples
get worse than the model really is.

**Where dropout goes (3 places):** on the attention weights (`Head`), after the head-mixing
projection (`MultiHeadAttention`), and at the end of the feed-forward (`FeedForward`). I've wired
the last two. The first — dropping attention *edges* — is yours.

*(I also finally added the **projection** to `MultiHeadAttention` that I promised in Step 1: the
learned layer that mixes the concatenated heads and gives the residual a clean path back.)*

**Your turn:** place the attention-weight dropout in `Head`, then run the dials + train cells.

In [27]:
# Step 5 hyperparameters — the dials. These OVERRIDE the small values from the setup cell.
# Start Mac-friendly; uncomment the FULL config for the ~1.5-loss run (slow on MPS, ideal on a GPU).
block_size    = 64      # context length      (was 8)  <- the single biggest lever
n_embd        = 128     # embedding width     (was 32)
n_layer       = 4       # transformer blocks  (was 3)
num_heads     = 4       # heads per block
batch_size    = 32      # sequences per batch (was 4)
dropout       = 0.1     # fraction of activations zeroed during training (was 0)

learning_rate = 3e-4    # gentler steps for a bigger model (was 1e-3)
max_iters     = 5000
eval_iters    = 200

# --- FULL config (~10M params, targets ~1.48 val loss). Slow on MPS (~15-30 min); ideal on Colab/GPU. ---
# block_size, n_embd, n_layer, num_heads, batch_size, dropout = 256, 384, 6, 6, 64, 0.2
# learning_rate, max_iters = 3e-4, 5000

print(f'config: block_size={block_size} n_embd={n_embd} n_layer={n_layer} num_heads={num_heads} '
      f'batch_size={batch_size} dropout={dropout} lr={learning_rate}')


config: block_size=64 n_embd=128 n_layer=4 num_heads=4 batch_size=32 dropout=0.1 lr=0.0003


In [28]:
# Step 5 — the complete, scaled architecture (all modules in one place, now with dropout).
# Mostly your own code from Steps 1-4, consolidated. The two additions: dropout (3 places) and the
# head-mixing projection in MultiHeadAttention (the "deferred mixing" promised back in Step 1).

class Head(nn.Module):
    """ One causal self-attention head, now with dropout on the attention weights. """
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)          # GIVEN: the dropout layer to use below
        self.head_size = head_size

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x); v = self.value(x)
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        # TODO(human): apply self.dropout to the attention weights `wei` HERE -- after softmax,
        #   before `wei @ v`. This randomly severs some token->token attention edges each training
        #   step, so the model can't over-rely on any single connection. One line: wei = self.dropout(wei)
        wei = self.dropout(wei)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """ Parallel heads, concatenated, then MIXED by a projection (+ dropout). """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)    # NEW: the learned mixing of the concatenated heads
        self.dropout = nn.Dropout(dropout)          # NEW
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))          # mix the heads, then dropout
        return out


class FeedForward(nn.Module):
    """ Per-token MLP, now with dropout as the final step. """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),                    # NEW: dropout on the way out
        )
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Communicate then compute, each with a residual + pre-norm (unchanged from Step 3). """
    def __init__(self, n_embd, num_heads):
        super().__init__()
        self.sa   = MultiHeadAttention(num_heads, n_embd // num_heads)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    """ The whole thing: embeddings -> stack of blocks -> final norm -> decode. """
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, num_heads) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [30]:
# Step 5 — train the scaled model.
torch.manual_seed(1337)
model = GPTLanguageModel().to(device)
print('params:', sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    if step % 500 == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\\n--- sample ---')
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))


params: 816705
step    0 | train 4.3352 | val 4.3307
step  500 | train 2.2918 | val 2.2938
step 1000 | train 2.0489 | val 2.0937
step 1500 | train 1.8954 | val 1.9759
step 2000 | train 1.7873 | val 1.8978
step 2500 | train 1.7173 | val 1.8681
step 3000 | train 1.6661 | val 1.8196
step 3500 | train 1.6228 | val 1.7860
step 4000 | train 1.5919 | val 1.7620
step 4500 | train 1.5580 | val 1.7349
\n--- sample ---


BRAYCA:
How you had with manise not dlam your pains ando:
Twere seep'd pamalleture; sign up thy cort,
For him Richards sween an ag; recond? aspeak Now Tears,
If on him; ere puerd you dredgent: I more our painot to knows
Are house me greet sir, Citendumanle't show
To names Where to us saburage:
FitZ to her hath his balis,
Iteal to unhoutreph'd life yourst ogatentweep;
There's loval so God you; for how pance.

SAPRUTH:
Where you, thanks, noned not pearsh, oven,
That I fore wift then
Honour him he windrow?

HENRY VI:
Blive; sutters done appyor homing worth olenest,
Mourts thin honou